In [0]:
%pip install -r ../requirements.txt
dbutils.library.restartPython()

In [0]:
import os
import sys

notebook_path = (dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().notebookPath().get())
repo_root = "/Workspace" + os.path.dirname(os.path.dirname(notebook_path))

sys.path.insert(0, repo_root)
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]

from src.preprocessing.build_dataset import load_modeling_sample
from src.preprocessing.features import CATEGORICAL_FEATURES, NUMERIC_FEATURES
from src.modeling.pipelines import build_model_pipeline
from src.evaluation.validation import split_by_municipality, cross_validate_grouped

spark.sql("USE CATALOG workspace")

sample_df = load_modeling_sample(spark, sample_fraction=0.15)
print(sample_df.shape)

train_df, test_df = split_by_municipality(sample_df)
feature_columns = CATEGORICAL_FEATURES + NUMERIC_FEATURES

for model_name in ["logistic", "hist_gradient_boosting"]:
    pipeline = build_model_pipeline(model_name)
    print(model_name)
    print(cross_validate_grouped(
        pipeline, train_df[feature_columns], train_df["target"], train_df["id_municipio"]
    ).round(3))

In [0]:
model_b_df = sample_df[sample_df["ano"] == 2024].copy()
train_b_df, test_b_df = split_by_municipality(model_b_df)

numeric_b = NUMERIC_FEATURES + ["previous_year_municipal_rate"]
features_b = CATEGORICAL_FEATURES + numeric_b

for model_name in ["logistic", "hist_gradient_boosting"]:
    pipeline = build_model_pipeline(model_name, numeric_features=numeric_b)
    print(model_name)
    print(cross_validate_grouped(
        pipeline, train_b_df[features_b], train_b_df["target"], train_b_df["id_municipio"]
    ).round(3))

In [0]:
population_df = spark.table("workspace.bronze.ibge_population").toPandas()
population_df["ano"] = population_df["ano"].astype("int64")

enriched_df = sample_df.merge(population_df, on=["ano", "id_municipio"], how="left")
print("missing population:", enriched_df["populacao"].isna().mean())

In [0]:
train_e_df, test_e_df = split_by_municipality(enriched_df)

numeric_e = NUMERIC_FEATURES + ["populacao"]
features_e = CATEGORICAL_FEATURES + numeric_e

for model_name in ["logistic", "hist_gradient_boosting"]:
    pipeline = build_model_pipeline(model_name, numeric_features=numeric_e)
    print(model_name)
    print(cross_validate_grouped(
        pipeline, train_e_df[features_e], train_e_df["target"], train_e_df["id_municipio"]
    ).round(3))

In [0]:
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, confusion_matrix

train_f_df, test_f_df = split_by_municipality(enriched_df)
features_f = CATEGORICAL_FEATURES + NUMERIC_FEATURES

final_pipeline = build_model_pipeline("logistic")
final_pipeline.fit(train_f_df[features_f], train_f_df["target"])

test_proba = final_pipeline.predict_proba(test_f_df[features_f])[:, 1]
test_pred = (test_proba >= 0.5).astype(int)

print("AUC:", round(roc_auc_score(test_f_df["target"], test_proba), 3))
print("F1:", round(f1_score(test_f_df["target"], test_pred), 3))
print("accuracy:", round(accuracy_score(test_f_df["target"], test_pred), 3))
print("baseline (always literate):", round(test_f_df["target"].mean(), 3))
print(confusion_matrix(test_f_df["target"], test_pred))

In [0]:
import numpy as np
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import balanced_accuracy_score, recall_score, precision_score

valid_proba = cross_val_predict(
    build_model_pipeline("logistic"),
    train_f_df[features_f], train_f_df["target"],
    groups=train_f_df["id_municipio"],
    cv=GroupKFold(n_splits=5),
    method="predict_proba",
)[:, 1]

thresholds = np.arange(0.40, 0.76, 0.05)
scores = [balanced_accuracy_score(train_f_df["target"], (valid_proba >= t).astype(int)) for t in thresholds]
best_threshold = thresholds[int(np.argmax(scores))]
print("best threshold:", round(best_threshold, 2))

test_pred_best = (test_proba >= best_threshold).astype(int)
print("balanced accuracy:", round(balanced_accuracy_score(test_f_df["target"], test_pred_best), 3))
print("recall not literate:", round(recall_score(test_f_df["target"], test_pred_best, pos_label=0), 3))
print("precision not literate:", round(precision_score(test_f_df["target"], test_pred_best, pos_label=0), 3))
print(confusion_matrix(test_f_df["target"], test_pred_best))